In [1]:
import pandas as pd
from pathlib import Path

# Folder containing the Believe .xlsx files (searched recursively, since
# they're split into '2024' and '2025' subfolders)
inputdirectory = '../../22 Fullstream Music/20260720. K Dillak and Dedjin files/Believe_ 2024 - 2025/CSV'
input_dir = Path(inputdirectory)

# Output goes one level above the input folder
output_dir = input_dir.parent
output_file = output_dir / "Believe_2024-2025_Combined_raw.xlsx"

# The actual data lives on this sheet in every file (the other sheet is just
# a 'Conversion Notes' explainer, not data)
data_sheet = "Feuil1"

frames = []
for file in sorted(input_dir.glob("**/*.xls*")):
    # Skip Excel's temporary lock files (e.g. '~$Q42025_Believe_Normalized.xlsx')
    # left behind when a workbook is open
    if file.name.startswith("~$"):
        continue

    df = pd.read_excel(file, sheet_name=data_sheet)
    df.insert(0, "File Name", file.name)
    frames.append(df)
    print(f"Read '{file.name}': {len(df)} rows")

if not frames:
    raise SystemExit(f"No .xls/.xlsx files found in {input_dir}")

# sort=False keeps column order stable; since all files share identical
# columns here, this is effectively just a straight row-wise concatenation
combined = pd.concat(frames, ignore_index=True, sort=False)

# The source files' headings are in French; translate them to English
french_to_english = {
    "Mois de Reporting": "Reporting Month",
    "Mois de vente": "Sales Month",
    "Plateforme": "Platform",
    "Pays / Région": "Country / Region",
    "Nom du label": "Label Name",
    "Nom de l'artiste": "Artist Name",
    "Titre de la sortie": "Release Title",
    "Titre de la piste": "Track Title",
    "UPC": "UPC",
    "ISRC": "ISRC",
    "Reference catalogue sortie": "Release Catalog Reference",
    "Type d'abonnement streaming": "Streaming Subscription Type",
    "Type de sortie": "Release Type",
    "Type de vente": "Sale Type",
    "Quantite": "Quantity",
    "Devise de paiement du client": "Customer Payment Currency",
    "Prix unitaire": "Unit Price",
    "Frais de reproduction mechanique": "Mechanical Reproduction Fee",
    "Revenu brut": "Gross Revenue",
    "Taux de revenu client": "Customer Revenue Rate",
    "Revenu Net": "Net Revenue",
    "Prix unitaire (CAD lisible)": "Unit Price (CAD readable)",
    "Revenu brut (CAD lisible)": "Gross Revenue (CAD readable)",
    "Taux de revenu client (%)": "Customer Revenue Rate (%)",
    "Revenu Net (CAD lisible)": "Net Revenue (CAD readable)",
}

untranslated = [col for col in combined.columns if col not in french_to_english and col != "File Name"]
if untranslated:
    print(f"WARNING: no English translation found for column(s): {untranslated}")

combined = combined.rename(columns=french_to_english)


def excel_serial_to_yyyymm(value):
    """Convert an Excel date-serial number (e.g. 45352, 45444) into a
    'YYYYMM' string. Values that aren't a plain number (already text,
    blank, etc.) are left untouched."""
    if pd.isna(value):
        return value
    try:
        serial = float(value)
    except (TypeError, ValueError):
        return value
    # Excel's day-0 is 1899-12-30 (this also matches its leap-year bug)
    date = pd.Timestamp("1899-12-30") + pd.Timedelta(days=serial)
    return date.strftime("%Y%m")


# 'Reporting Month' / 'Sales Month' sometimes come through as raw Excel date
# serials (e.g. 45352) instead of a readable month; normalise those to 'YYYYMM'
for month_col in ["Reporting Month", "Sales Month"]:
    combined[month_col] = combined[month_col].apply(excel_serial_to_yyyymm)

print(f"\nCombined: {len(combined)} rows, {len(combined.columns)} columns from {len(frames)} file(s)")

output_dir.mkdir(parents=True, exist_ok=True)
combined.to_excel(output_file, index=False)
print(f"Wrote combined file -> {output_file}")

Read 'Q12024_Believe_Normalized.xlsx': 6661 rows
Read 'Q22024_Believe_Normalized.xlsx': 22104 rows
Read 'Q32024_Believe_Normalized.xlsx': 30841 rows
Read 'Q42024_Believe_Normalized.xlsx': 77763 rows
Read 'Q12025_Believe_Normalized.xlsx': 61942 rows
Read 'Q22025_Believe_Normalized.xlsx': 82524 rows
Read 'Q32025_Believe_Normalized.xlsx': 73427 rows
Read 'Q42025_Believe_Normalized.xlsx': 71923 rows

Combined: 427185 rows, 26 columns from 8 file(s)
Wrote combined file -> ../../22 Fullstream Music/20260720. K Dillak and Dedjin files/Believe_ 2024 - 2025/Believe_2024-2025_Combined_raw.xlsx
